# LeetCode 3668 — Restore Finishing Order bằng Python

Link đề: https://leetcode.com/problems/restore-finishing-order/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def recoverOrder(self, order: List[int], friends: List[int]) -> List[int]:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: `order` là hoán vị của `1..n` liệt kê **ID người chạy theo thứ tự về đích**; `friends` là các ID bạn bè (đã sắp tăng dần). Trả về ID của bạn bè **theo thứ tự về đích**.
2. Phân biệt **ID** (giá trị) với **vị trí** (chỉ số trong mảng) — nguồn gốc của nhiều lỗi nhầm lẫn.
3. Biết dùng **tập hợp (`set`)** để kiểm tra "thuộc `friends`?" trong `O(1)`, và **từ điển vị trí** `pos[id] = index` để sắp xếp theo thứ tự về đích.
4. Viết 4 cách (kiểm tra thuộc danh sách, `set`, sắp xếp theo vị trí, mảng boolean theo ID) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: trả `friends` nguyên trạng, trả **vị trí** thay vì ID, sửa danh sách khi đang duyệt, và dùng ID làm chỉ số mảng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `order` độ dài `n` và mảng số nguyên `friends`.

- `order` là một **hoán vị** của các số từ `1` đến `n`. Mỗi giá trị là **ID** của một người tham gia, xếp theo thứ tự họ **về đích**.
- `friends` chứa ID các **bạn bè** của bạn, được sắp **tăng ngặt**. Mọi ID trong `friends` đều có trong `order`.

Trả về mảng ID các bạn bè, **sắp xếp theo thứ tự họ về đích**.

| Ví dụ | order | friends | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[3,1,2,5,4]` | `[1,3,4]` | `[3,1,4]` | thứ tự về đích 3, 1, 2, 5, 4; bạn bè (1, 3, 4) về theo thứ tự 3, 1, 4 |
| 2 | `[1,4,5,3,2]` | `[2,5]` | `[5,2]` | thứ tự về đích 1, 4, 5, 3, 2; bạn bè (2, 5) về theo thứ tự 5, 2 |

Constraints: `1 <= n == order.length <= 100`; `order` chứa mỗi số `1..n` đúng một lần; `1 <= friends.length <= min(8, n)`; `1 <= friends[i] <= n`; `friends` tăng ngặt.

In [1]:
order = [3, 1, 2, 5, 4]
friends = [1, 3, 4]
result = []
for runner in order:                 # đi theo thứ tự về đích
    if runner in friends:            # nếu là bạn bè thì ghi lại
        result.append(runner)
print(result)
order, friends = [1, 4, 5, 3, 2], [2, 5]
print([x for x in order if x in friends])

[3, 1, 4]
[5, 2]


## 3. Kiến thức nền

### 3.1. Hoán vị, ID và vị trí

`order = [3,1,2,5,4]` nghĩa là: người về **thứ 1** (chỉ số `0`) có ID `3`; về thứ 2 có ID `1`; ... **ID** là giá trị `order[i]`, **vị trí** là chỉ số `i`. Hai khái niệm này khác nhau: ID `3` có vị trí `0`. Đề trả về **ID**, không phải vị trí.

In [2]:
order = [3, 1, 2, 5, 4]
for i, runner in enumerate(order):
    print(f"vị trí {i}: ID {runner}")

vị trí 0: ID 3
vị trí 1: ID 1
vị trí 2: ID 2
vị trí 3: ID 5
vị trí 4: ID 4


### 3.2. `set` — kiểm tra thuộc trong `O(1)`

`x in list` phải so sánh tuần tự (`O(m)`), còn `x in set` dùng băm (`O(1)` trung bình). `set(friends)` chỉ tạo một lần rồi dùng cho mọi phần tử của `order`. Cần nhớ: `set` **không giữ thứ tự** nên chỉ dùng để *kiểm tra*, không dùng để *xuất kết quả*.

In [3]:
friends = [1, 3, 4]
fs = set(friends)
print(fs, 3 in fs, 2 in fs)
print([x for x in [3, 1, 2, 5, 4] if x in fs])      # thứ tự lấy từ order, không phải từ set

{1, 3, 4} True False
[3, 1, 4]


### 3.3. Từ điển vị trí và `sorted(..., key=...)`

Tạo `pos = {id: index}` bằng `enumerate`. Sau đó `sorted(friends, key=pos.get)` (hay `key=lambda f: pos[f]`) sắp xếp bạn bè theo **vị trí về đích**. Đây là cách "sắp xếp theo thứ hạng cho trước".

In [4]:
order = [3, 1, 2, 5, 4]
pos = {runner: i for i, runner in enumerate(order)}
print(pos)
print(sorted([1, 3, 4], key=lambda f: pos[f]))

{3: 0, 1: 1, 2: 2, 5: 3, 4: 4}
[3, 1, 4]


### 3.4. Mảng boolean theo ID (counting array)

Vì ID nằm trong `1..n`, ta dùng mảng `is_friend = [False] * (n + 1)` và `is_friend[f] = True`. Kiểm tra bằng chỉ số `O(1)`, không cần `set`. Ở đây **ID dùng làm chỉ số** có chủ đích (khác với lỗi nhầm ở mục 13): ta đặt cờ `True` tại chỉ số bằng ID.

In [5]:
n = 5
is_friend = [False] * (n + 1)          # chỉ số 0 không dùng vì ID bắt đầu từ 1
for f in [1, 3, 4]:
    is_friend[f] = True
print(is_friend)
print([x for x in [3, 1, 2, 5, 4] if is_friend[x]])

[False, True, False, True, True, False]
[3, 1, 4]


### 3.5. List comprehension có điều kiện và `enumerate`

`[x for x in order if x in fs]` là cách viết gọn của vòng `for` + `if` + `append`. `enumerate(order)` cho cặp `(vị trí, ID)`; `enumerate(order, start=1)` đánh số từ `1`. Hai công cụ này đủ để giải mọi cách trong notebook.

In [6]:
order = [3, 1, 2, 5, 4]
fs = {1, 3, 4}
loop_result = []
for x in order:
    if x in fs:
        loop_result.append(x)
comp_result = [x for x in order if x in fs]
print(loop_result == comp_result, comp_result)
print(list(enumerate(order)), list(enumerate(order, start=1))[:2])
print([(i, x) for i, x in enumerate(order, start=1) if x in fs])

True [3, 1, 4]
[(0, 3), (1, 1), (2, 2), (3, 5), (4, 4)] [(1, 3), (2, 1)]
[(1, 3), (2, 1), (5, 4)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung cuộc đua: trọng tài đọc tên người về đích lần lượt `3, 1, 2, 5, 4`. Bạn chỉ **ghi lại** tên khi nghe đúng người thân (`1, 3, 4`):

```
nghe 3 -> là bạn  -> ghi: [3]
nghe 1 -> là bạn  -> ghi: [3, 1]
nghe 2 -> không   -> bỏ qua
nghe 5 -> không   -> bỏ qua
nghe 4 -> là bạn  -> ghi: [3, 1, 4]
```

Danh sách ghi được **đã đúng thứ tự về đích** vì ta đọc theo `order`. Không cần sắp xếp gì cả!

Ghi nhớ: duyệt **`order`** (nguồn có thứ tự) và **lọc** bằng `friends` (nguồn chỉ dùng để tra cứu), chứ đừng duyệt `friends` rồi tìm vị trí.

## 5. Cách 1 — Brute Force / baseline

Duyệt `order` từ đầu đến cuối; với mỗi `x` kiểm tra `x in friends` (tìm tuần tự trong **danh sách**) rồi thêm vào kết quả.

Nút thắt: `x in friends` tốn `O(m)` với `m = len(friends) <= 8`, nên tổng `O(n * m)`. Với `n <= 100`, `m <= 8` thì chỉ ~800 phép so sánh, không đáng kể, nhưng khi `friends` lớn thì chậm — dùng `set` giải quyết.

In [7]:
class SolutionBrute:
    def recoverOrder(self, order, friends):
        result = []
        for x in order:
            if x in friends:           # tìm tuần tự trong list: O(len(friends))
                result.append(x)
        return result

print(SolutionBrute().recoverOrder([3, 1, 2, 5, 4], [1, 3, 4]))
print(SolutionBrute().recoverOrder([1, 4, 5, 3, 2], [2, 5]))

[3, 1, 4]
[5, 2]


**Độ phức tạp:** `n = len(order)`, `m = len(friends)`. Time `O(n * m)`, Space `O(m)` cho kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** vì `order` **đã được sắp** theo thời gian về đích, kết quả chỉ là **dãy con** của `order` gồm các phần tử thuộc `friends`. Chỉ cần một lượt quét và một phép kiểm tra thuộc rẻ (`set`).

**Quy tắc "nếu… thì…":**

- **Nếu** `x` (từ `order`) thuộc tập bạn bè **thì** thêm `x` vào kết quả.
- **Nếu không** thì bỏ qua.
- Kết quả giữ nguyên thứ tự gặp trong `order`.

Invariant: sau khi xử lý `k` phần tử đầu của `order`, `result` chứa đúng các bạn bè trong `k` phần tử đó theo thứ tự về đích.

In [8]:
def recover(order, friends):
    fs = set(friends)
    return [x for x in order if x in fs]

print(recover([3, 1, 2, 5, 4], [1, 3, 4]), recover([1, 4, 5, 3, 2], [2, 5]))

[3, 1, 4] [5, 2]


## 7. Các cách tiếp theo

### Cách 2 — Lọc `order` bằng `set` (cách nộp bài)

Tạo `set(friends)` một lần, duyệt `order` và giữ những `x` thuộc `set`. Ngắn gọn bằng list comprehension.

In [9]:
class SolutionSet:
    def recoverOrder(self, order, friends):
        fs = set(friends)
        return [x for x in order if x in fs]

print(SolutionSet().recoverOrder([3, 1, 2, 5, 4], [1, 3, 4]), SolutionSet().recoverOrder([1, 4, 5, 3, 2], [2, 5]))

[3, 1, 4] [5, 2]


**Độ phức tạp:** Time `O(n + m)`, Space `O(m)`.

### Cách 3 — Sắp xếp bạn bè theo vị trí về đích

Lập `pos[id] = index` cho cả `order` rồi `sorted(friends, key=pos.__getitem__)`. Cách này nghĩ theo hướng "sắp xếp các bạn theo thứ hạng", hữu ích khi `friends` ít còn `order` rất dài và bạn đã có sẵn bảng vị trí.

In [10]:
class SolutionRank:
    def recoverOrder(self, order, friends):
        pos = {runner: i for i, runner in enumerate(order)}
        return sorted(friends, key=lambda f: pos[f])

print(SolutionRank().recoverOrder([3, 1, 2, 5, 4], [1, 3, 4]), SolutionRank().recoverOrder([1, 4, 5, 3, 2], [2, 5]))

[3, 1, 4] [5, 2]


**Độ phức tạp:** Time `O(n + m log m)` (dựng `pos` rồi sắp `m` phần tử), Space `O(n)` cho `pos`.

### Cách 4 — Mảng boolean theo ID

Vì ID nằm trong `1..n`, dùng mảng cờ `is_friend[id]`. Mỗi lần kiểm tra là truy cập mảng `O(1)`, không băm.

In [11]:
class SolutionFlags:
    def recoverOrder(self, order, friends):
        n = len(order)
        is_friend = [False] * (n + 1)
        for f in friends:
            is_friend[f] = True
        result = []
        for x in order:
            if is_friend[x]:
                result.append(x)
        return result

print(SolutionFlags().recoverOrder([3, 1, 2, 5, 4], [1, 3, 4]), SolutionFlags().recoverOrder([1, 4, 5, 3, 2], [2, 5]))

[3, 1, 4] [5, 2]


**Độ phức tạp:** Time `O(n + m)`, Space `O(n)` cho mảng cờ.

## 8. TRACE từng bước

Trace `order = [3,1,2,5,4]`, `friends = [1,3,4]` (tập `{1,3,4}`).

| i | x = order[i] | x thuộc tập? | result sau bước |
|---|---|---|---|
| 0 | 3 | có | `[3]` |
| 1 | 1 | có | `[3, 1]` |
| 2 | 2 | không | `[3, 1]` |
| 3 | 5 | không | `[3, 1]` |
| 4 | 4 | có | `[3, 1, 4]` |

In [12]:
def trace_recover(order, friends):
    fs = set(friends)
    result = []
    print("order =", order, "| friends =", friends)
    for i, x in enumerate(order):
        keep = x in fs
        if keep:
            result.append(x)
        print(f"i={i}: x={x} thuộc friends? {keep!s:<5} -> result={result}")
    return result

trace_recover([3, 1, 2, 5, 4], [1, 3, 4])
trace_recover([1, 4, 5, 3, 2], [2, 5])

order = [3, 1, 2, 5, 4] | friends = [1, 3, 4]
i=0: x=3 thuộc friends? True  -> result=[3]
i=1: x=1 thuộc friends? True  -> result=[3, 1]
i=2: x=2 thuộc friends? False -> result=[3, 1]
i=3: x=5 thuộc friends? False -> result=[3, 1]
i=4: x=4 thuộc friends? True  -> result=[3, 1, 4]
order = [1, 4, 5, 3, 2] | friends = [2, 5]
i=0: x=1 thuộc friends? False -> result=[]
i=1: x=4 thuộc friends? False -> result=[]
i=2: x=5 thuộc friends? True  -> result=[5]
i=3: x=3 thuộc friends? False -> result=[5]
i=4: x=2 thuộc friends? True  -> result=[5, 2]


[5, 2]

## 9. Code nộp LeetCode

In [13]:
from typing import List, Optional

class Solution:
    def recoverOrder(self, order: List[int], friends: List[int]) -> List[int]:
        friend_set = set(friends)
        return [runner for runner in order if runner in friend_set]

## 10. Vì sao thuật toán đúng

- **Đủ và đúng phần tử:** mỗi bạn bè xuất hiện trong `order` **đúng một lần** (`order` là hoán vị, mọi ID của `friends` đều có trong `order`). Ta giữ mọi `x` thuộc `friends`, nên kết quả có đủ mọi bạn bè, không lặp, không có người lạ.
- **Đúng thứ tự:** kết quả được tạo bằng cách quét `order` từ trái sang phải và chỉ thêm vào cuối, nên thứ tự tương đối giữ nguyên chính là thứ tự về đích (`order[i]` về trước `order[j]` khi `i < j`).
- **Không phụ thuộc tính sắp xếp của `friends`:** `friends` chỉ dùng để tra cứu thành viên; đề cho nó đã sắp tăng dần nhưng thuật toán không cần điều đó.
- **Độ dài kết quả bằng `len(friends)`**, nên cũng có thể dùng làm kiểm tra phụ (`assert`).

## 11. Phân tích Time / Space Complexity

`n = len(order) <= 100`, `m = len(friends) <= min(8, n)`.

| Cách | Time | Space |
|---|---|---|
| Kiểm tra thuộc `list` | `O(n * m)` | `O(m)` |
| Lọc bằng `set` | `O(n + m)` | `O(m)` |
| Sắp xếp theo vị trí | `O(n + m log m)` | `O(n)` |
| Mảng cờ theo ID | `O(n + m)` | `O(n)` |

Trong giới hạn của đề mọi cách đều rất nhanh. Cách `set` là cân bằng nhất giữa độ ngắn và hiệu quả.

## 12. Bộ test

Ví dụ của đề, biên (`n = 1`, `friends` là tất cả mọi người, `friends` chỉ một người) và đối chiếu **tất cả cách** trên input ngẫu nhiên tuân thủ constraints.

In [14]:
impls = [SolutionBrute(), SolutionSet(), SolutionRank(), SolutionFlags(), Solution()]

cases = [
    ([3, 1, 2, 5, 4], [1, 3, 4], [3, 1, 4]),
    ([1, 4, 5, 3, 2], [2, 5], [5, 2]),
    ([1], [1], [1]),                              # n = 1
    ([2, 1], [1, 2], [2, 1]),                     # tất cả đều là bạn: kết quả = order
    ([4, 3, 2, 1], [3], [3]),                     # một bạn
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], [2, 4, 6, 8], [2, 4, 6, 8]),   # order đã tăng -> kết quả = friends
    ([9, 8, 7, 6, 5, 4, 3, 2, 1], [2, 4, 6, 8], [8, 6, 4, 2]),   # order giảm -> đảo friends
]
for order, friends, exp in cases:
    for impl in impls:
        got = impl.recoverOrder(order[:], friends[:])
        assert got == exp, (type(impl).__name__, order, friends, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 7 ca


In [15]:
import random
random.seed(3668)

def gen_case():
    n = random.randint(1, 100)
    order = list(range(1, n + 1))
    random.shuffle(order)
    m = random.randint(1, min(8, n))
    friends = sorted(random.sample(range(1, n + 1), m))      # tăng ngặt, không trùng
    return order, friends

for _ in range(1500):
    order, friends = gen_case()
    results = [impl.recoverOrder(order[:], friends[:]) for impl in impls]
    first = results[0]
    assert all(r == first for r in results)
    assert sorted(first) == friends                           # đủ mọi bạn bè, không lặp
    idx = [order.index(x) for x in first]
    assert idx == sorted(idx)                                 # vị trí tăng dần = đúng thứ tự về đích
print("1500 test ngẫu nhiên (seed=3668) OK")

1500 test ngẫu nhiên (seed=3668) OK


In [16]:
# Hàm không được làm hỏng input
order, friends = [3, 1, 2, 5, 4], [1, 3, 4]
o2, f2 = order[:], friends[:]
Solution().recoverOrder(order, friends)
assert order == o2 and friends == f2
# Độ dài kết quả luôn bằng len(friends)
for _ in range(200):
    order, friends = gen_case()
    assert len(Solution().recoverOrder(order, friends)) == len(friends)
print("không sửa input + độ dài đúng: OK")

không sửa input + độ dài đúng: OK


### Kiểm tra độc lập bằng "vị trí tăng dần"

Một đáp án đúng phải thỏa hai điều: (1) là **hoán vị** của `friends`; (2) các **vị trí** trong `order` của nó **tăng ngặt**. Hàm `is_valid_answer` dưới đây kiểm tra hai điều đó mà **không** dùng thuật toán giải — dùng để bắt lỗi nếu ta lỡ sai cả bản giải lẫn bản đối chiếu.

In [17]:
def is_valid_answer(order, friends, ans):
    if sorted(ans) != sorted(friends):
        return False
    positions = [order.index(x) for x in ans]
    return all(a < b for a, b in zip(positions, positions[1:]))

assert is_valid_answer([3, 1, 2, 5, 4], [1, 3, 4], [3, 1, 4])
assert not is_valid_answer([3, 1, 2, 5, 4], [1, 3, 4], [1, 3, 4])
assert not is_valid_answer([3, 1, 2, 5, 4], [1, 3, 4], [3, 1])
for _ in range(500):
    order, friends = gen_case()
    assert is_valid_answer(order, friends, Solution().recoverOrder(order, friends))
print("is_valid_answer: 500 ca OK")

is_valid_answer: 500 ca OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: trả `friends` nguyên trạng (hoặc sắp theo ID)

`friends` đã tăng dần theo **ID**, nhưng đề yêu cầu theo **thứ tự về đích**.

In [18]:
def wrong_return_friends(order, friends):
    return friends                      # SAI: bỏ qua order

order, friends = [3, 1, 2, 5, 4], [1, 3, 4]
print(wrong_return_friends(order, friends), "!=", Solution().recoverOrder(order, friends))
assert wrong_return_friends(order, friends) != Solution().recoverOrder(order, friends)   # [1,3,4] != [3,1,4]

[1, 3, 4] != [3, 1, 4]


### Lỗi 2: trả **vị trí** thay vì ID

`order.index(f)` là **vị trí** về đích, không phải ID. Đề muốn ID.

In [19]:
def wrong_positions(order, friends):
    return sorted(order.index(f) for f in friends)       # SAI: trả chỉ số

order, friends = [1, 4, 5, 3, 2], [2, 5]
print(wrong_positions(order, friends), "!=", Solution().recoverOrder(order, friends))
assert wrong_positions(order, friends) != Solution().recoverOrder(order, friends)        # [2,4] != [5,2]

[2, 4] != [5, 2]


### Lỗi 3: xóa phần tử khỏi danh sách khi đang duyệt nó

Vừa duyệt `order` vừa `remove` làm các chỉ số dịch chuyển, nên **bỏ sót** phần tử.

In [20]:
def wrong_remove_while_iterating(order, friends):
    lst = order[:]                       # copy để không sửa input, nhưng vẫn duyệt chính lst
    for x in lst:
        if x not in friends:
            lst.remove(x)                # SAI: sửa lst khi đang for qua lst
    return lst

order, friends = [3, 1, 2, 5, 4], [1, 3, 4]
print(wrong_remove_while_iterating(order, friends), "!=", Solution().recoverOrder(order, friends))
assert wrong_remove_while_iterating(order, friends) != Solution().recoverOrder(order, friends)   # [3,1,5,4] != [3,1,4]

[3, 1, 5, 4] != [3, 1, 4]


### Lỗi 4: nhầm ID với chỉ số mảng khi sắp xếp

`order[f]` là **ID ở vị trí `f`**, không phải **vị trí của ID `f`**. Muốn vị trí của `f` phải dùng `order.index(f)` hoặc từ điển `pos`.

In [21]:
def wrong_id_as_index(order, friends):
    return sorted(friends, key=lambda f: order[f])       # SAI: order[f] không phải vị trí của f

order, friends = [3, 1, 2, 5, 4], [1, 3, 4]
print(wrong_id_as_index(order, friends), "!=", Solution().recoverOrder(order, friends))
assert wrong_id_as_index(order, friends) != Solution().recoverOrder(order, friends)              # [1,4,3] != [3,1,4]

[1, 4, 3] != [3, 1, 4]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Kết quả là "**dãy con** của một dãy đã có thứ tự, giữ các phần tử thỏa điều kiện" -> **duyệt dãy có thứ tự rồi lọc**, không cần sắp xếp.
- Nhiều lần hỏi "x có thuộc tập K không" -> `set(K)`; khóa là số nguyên nhỏ liên tục -> mảng cờ.
- Phân biệt **giá trị (ID)** và **chỉ số (vị trí)**; ghi rõ trong đầu "mình cần cái nào?".
- Đừng sửa danh sách đang `for`; hãy tạo danh sách kết quả mới.
- Khi đề đã cho `friends` sắp tăng dần: thường chỉ để **đảm bảo không trùng**, không phải mẹo giải.
- `n <= 100` nên mọi cách đều qua; ưu tiên cách **dễ đúng** nhất (list comprehension với `set`).

## 15. Biến thể của bài

1. **Intersection of Two Arrays** (LeetCode 349): giao hai mảng (không cần thứ tự).
2. **Relative Sort Array** (1122): sắp `arr1` theo thứ tự cho trước trong `arr2`.
3. Cho `friends` **không** sắp tăng và có thể **trùng** ID: cần khử trùng bằng `set` trước.
4. Trả về **thứ hạng** (vị trí 1-indexed) của từng bạn bè theo thứ tự ID tăng dần.
5. Chỉ trả `k` bạn bè về đích **sớm nhất** (cắt `[:k]` sau khi lọc).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Kiểm tra thuộc `list` | `x in friends` | `O(n m)` | `O(m)` | `friends` rất nhỏ, viết nhanh |
| Lọc bằng `set` | `x in set(friends)` | `O(n + m)` | `O(m)` | Cách nộp chuẩn |
| Sắp theo vị trí | `pos` rồi `sorted(key=pos)` | `O(n + m log m)` | `O(n)` | Khi đã có bảng thứ hạng |
| Mảng cờ | `is_friend[id]` | `O(n + m)` | `O(n)` | ID là số nguyên nhỏ liên tục |

Chọn lọc bằng `set` để nộp; dùng sắp xếp theo vị trí làm bản đối chiếu độc lập.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().recoverOrder([2, 3, 1], [1, 2])` là gì?

**Bài 2 (tính tay):** `order = [5,3,1,2,4]`, `friends = [1,4,5]`: ghi từng bước.

**Bài 3 (điền TODO):** hoàn thành `build_pos(order)` trả từ điển `{ID: vị trí}`.

**Bài 4 (tự viết lại):** viết `finishing_ranks(order, friends)` trả **vị trí 1-indexed** của từng bạn bè theo thứ tự về đích.

**Bài 5 (mini challenge):** viết `first_k_friends(order, friends, k)` trả `k` bạn bè về đích sớm nhất (nếu ít hơn `k` thì trả hết).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().recoverOrder([2, 3, 1], [1, 2]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def build_pos_todo(order):
    pos = {}
    # TODO: dùng enumerate(order) để gán pos[ID] = vị trí
    return pos

[2, 1]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: đọc `order` từ trái sang phải, ghi lại những ID thuộc `{1, 2}`. Bài 2: bạn bè là `1`, `4`, `5`.

**Mức 2 (vừa):** Bài 3: `{runner: i for i, runner in enumerate(order)}`. Bài 4: duyệt `order` với `enumerate(order, start=1)` và chọn các ID thuộc `friends`. Bài 5: lọc rồi cắt lát `[:k]`.

**Mức 3 (gần đáp án):** Bài 1: `[2, 1]`. Bài 2: thứ tự về đích 5, 3, 1, 2, 4 -> bạn bè về theo 5, 1, 4. Bài 4: kết quả là danh sách vị trí `i` (1-indexed) của các bạn bè, tăng dần.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
assert Solution().recoverOrder([2, 3, 1], [1, 2]) == [2, 1]

# Bài 2
assert Solution().recoverOrder([5, 3, 1, 2, 4], [1, 4, 5]) == [5, 1, 4]

# Bài 3
def build_pos(order):
    return {runner: i for i, runner in enumerate(order)}
assert build_pos([3, 1, 2]) == {3: 0, 1: 1, 2: 2}

# Bài 4
def finishing_ranks(order, friends):
    fs = set(friends)
    return [i for i, x in enumerate(order, start=1) if x in fs]
assert finishing_ranks([3, 1, 2, 5, 4], [1, 3, 4]) == [1, 2, 5]
assert finishing_ranks([1, 4, 5, 3, 2], [2, 5]) == [3, 5]

# Bài 5
def first_k_friends(order, friends, k):
    return Solution().recoverOrder(order, friends)[:k]
assert first_k_friends([3, 1, 2, 5, 4], [1, 3, 4], 2) == [3, 1]
assert first_k_friends([3, 1, 2, 5, 4], [1, 3, 4], 10) == [3, 1, 4]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "lọc dãy có thứ tự theo một tập":**

```python
keep = set(friends)                       # tập tra cứu
return [x for x in order if x in keep]    # duyệt dãy có thứ tự, giữ thứ tự
```

**Checklist trước khi submit:**

- [ ] Duyệt **`order`** (có thứ tự về đích), không duyệt `friends`?
- [ ] Trả về **ID**, không phải vị trí?
- [ ] Không sửa danh sách đang duyệt?
- [ ] Dùng `set` (hoặc mảng cờ) để kiểm tra thuộc cho nhanh?
- [ ] Đã thử `n = 1`, tất cả đều là bạn, `friends` một người?
- [ ] Chữ ký đúng `recoverOrder(self, order: List[int], friends: List[int]) -> List[int]`?